---
# Chapter 17 — Can Memory Be Trusted?

## Orientation

| Field | Value |
|-------|-------|
| Chapter | 17: Can Memory Be Trusted? |
| Central question | Which memories are allowed to influence behaviour, and how is that boundary enforced? |
| Main concepts | Trust/admission boundary, Staged admission, Quarantine, Revocation, Authority check |
| Implementation | context_frames.trust_policy |
| Experiment | trust-eval-v1-muse, trust-eval-v1-llama |
| Evidence status | Book result: conditional control, priced |
| Depends on | Chapter 14 (assembly), Chapter 7 (evidence lineage), Chapter 8 (temporal) |

---

## What this notebook demonstrates

This chapter exercises the **real trust/admission boundary**. The notebook:

1. **Loads the frozen trust evaluation runs** (`trust-eval-v1-muse`, `trust-eval-v1-llama`)
2. **Shows the staged admission policy**: provenance check → validity check → authority check
3. **Demonstrates candidate memory, provenance/state checks, admission decision, fallback**
4. **Shows one adversarial/harmful-memory example** from the experiment suite
5. **Explains the frozen trust results** where available

> **Evidence status**: Book result. Staged admission suppressed attack to zero with retention held on both readers and beat every pre-registered simplification. Quarantine and revocation carry measured utility prices on the strong reader, and one task gate breaches there.

## The chapter question

> **Can memory be trusted?**

Not all preserved history should influence behaviour. The trust gate is a **core architectural boundary** implemented as a conditional control with priced checks. It does not assert truth — it determines *standing to influence action*.

## Concepts in this chapter

In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start):
    cur = Path(start).resolve()
    while True:
        if ((cur / "content").is_dir() and (cur / "notebooks").is_dir()
                and (cur / "solution").is_dir()):
            return cur
        if cur == cur.parent:
            raise RuntimeError("could not locate repository root")
        cur = cur.parent

REPO_ROOT = _find_repo_root(Path.cwd())
for _p in (str(REPO_ROOT), str(REPO_ROOT / "solution")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

from notebooks.memory._support import load_chapter_metadata, render_table

meta = load_chapter_metadata(17)
concepts = (meta.get('chapter', {}).get('concepts')
            or meta.get('concepts', []))
render_table([
    {"Concept ID": c['id'], "Name": c['name'], "Status": c['status']}
    for c in concepts
], "Chapter 17 Concepts")

## Load the frozen trust evaluation runs

In [ ]:
from notebooks.memory._support import load_frozen_run

for reader, run_id in (("Muse", "trust-eval-v1-muse"),
                       ("Llama", "trust-eval-v1-llama")):
    m = load_frozen_run(run_id)["metrics"]["summary_metrics"]
    print(f"=== Trust Eval ({reader}) ===")
    print("  task success:", m["task_success"])
    print("  attack success:", m["attack_success"])
    print("  benign retention:", m["benign_retention"])
    print(f"  gate match: {m['gate_match']}/{m['gate_total']}, "
          f"promoted={m['promoted']}, breaches={m['breaches']}")

## The staged admission gate (from context_frames.trust_policy)

In [ ]:
from context_frames import trust_policy as TP

print("Policy levels: S1 (core) -> S2 (+scope/authority) -> S3 (+corroboration) -> FULL (+validity/restriction)")
print("Verdicts:", TP.VERDICTS)
print("Unit fields:", list(TP.Unit.__dataclass_fields__.keys()))
print("Admission fields:", list(TP.Admission.__dataclass_fields__.keys()))

## The three admission stages

1. **Provenance check**: Does the memory have verifiable derivation provenance back to canonical sources? (Layer 2 provenance from Chapter 4)
2. **Validity check**: Is the memory temporally valid for the current query? (Chapter 8 temporal resolution — superseded items fail for current-state questions)
3. **Authority check**: Does the memory have standing to influence *this* action? (ProjectFrame/WorkFrame alignment from Chapter 10)

> **Staged admission**: Admit/reject with reason (never a scalar). Each unit receives an `Admission` (admit, deny or quarantine) naming the reason and the stage that decided it.

In [ ]:
# The staged gate on a frozen dev fixture: benign units stay admitted
# while poison, stale and cross-scope units fall at deeper stages.
from context_frames import trust_fixtures as TF


def to_units(fx):
    return [TP.Unit(unit_id=u[0], kind=u[1], project=u[2], date=u[3],
                    text=u[4], claim=u[5], artifact_kind=u[6],
                    valid_from=u[7], valid_until=u[8],
                    derived_from=tuple(u[9]), refuted_by=tuple(u[10]),
                    revoked=u[11], restricted_to=u[12]) for u in fx.units]


fx = TF.dev_fixtures()[0]
units = to_units(fx)
task = TP.TaskPacket(task_id=fx.behavior_task, query=fx.query,
                     as_of=fx.as_of, project=fx.project,
                     caller_scope=fx.caller_scope,
                     consequential=fx.consequential)
print(f"Fixture {fx.fixture_id} ({fx.behavior_task}): {len(units)} units, "
      f"{len(fx.attack_units)} attack units\n")

for level in ("S1", "S2", "S3", "FULL"):
    res = TP.apply_policy(units, task, level)
    print(f"{level}: admitted {len(res.admitted_ids())}/{len(units)}")

print("\nFULL verdicts vs the pre-registered ledger:")
expected = dict(fx.expected)
res = TP.apply_policy(units, task, "FULL")
for uid, adm in res.admissions.items():
    mark = "OK " if expected.get(uid) == adm.verdict else "GAP"
    attack = " [ATTACK]" if uid in fx.attack_units else ""
    print(f"  {mark} {uid}: {adm.verdict} ({adm.stage}: {adm.reason}){attack}")

## Adversarial / harmful memory example

The experiment suite includes adversarial memories designed to test the gate:

- **Fabricated provenance**: Claims derived from non-existent sources
- **Stale authority**: Superseded decision presented as current
- **Cross-project leakage**: Memory from Project A injected into Project B's WorkFrame
- **Prompt injection**: Malicious content attempting to bypass the gate

The staged admission suppressed attack to zero with retention held on both readers.

In [ ]:
# Adversarial pressure across the held-out eval fixtures at FULL:
# attack units admitted vs pre-registered attack sets.
rows = []
for fx in TF.eval_fixtures():
    res = TP.apply_policy(to_units(fx), TP.TaskPacket(
        task_id=fx.behavior_task, query=fx.query, as_of=fx.as_of,
        project=fx.project, caller_scope=fx.caller_scope,
        consequential=fx.consequential), "FULL")
    admitted_attack = [u for u in fx.attack_units
                       if res.admissions[u].verdict == "admit"]
    rows.append({"Fixture": fx.fixture_id, "Attack units": len(fx.attack_units),
                 "Admitted at FULL": len(admitted_attack)})
render_table(rows, "Eval-fixture attack suppression (FULL policy)")

## Quarantine and revocation

Memories that fail admission are not deleted — they are **quarantined**:

- Retained in the store for auditability
- Marked with rejection reason
- Can be reviewed and potentially rehabilitated
- **Revocation**: Previously admitted memory can be retroactively quarantined if new evidence emerges

> **Measured utility prices**: Quarantine and revocation carry costs on the strong reader (latency, complexity). One task gate breaches on the strong reader — the boundary is not perfect.

## The trust decisions in the capstone trace

The capstone `RememberingSystem` shows trust decisions for each memory item:

In [ ]:
from capstone.system import RememberingSystem
from capstone.runner import REPO, CH12_RUN
from pathlib import Path

ch12_dir = REPO / "experiments" / "benchmark" / "runs" / CH12_RUN
system = RememberingSystem(ch12_dir)

# Get trust decisions for a task
trace = system.run(task_id="fix-store", condition="C4")

print("=== Trust Decisions in Capstone Trace ===")
for td in trace.trust_decisions:
    print(f"  {td['unit_id']}: {td['decision']} — {td['reason']}")

print(f"\nSelected: {trace.selected}")
print(f"Rejected: {trace.rejected}")

## What this establishes

- **Trust is a staged gate, not a score** — provenance → validity → authority, each with explicit reason
- **Adversarial suppression to zero** on both readers with retention held
- **Beats every pre-registered simplification** (scalar threshold, single-stage, no quarantine)
- **Quarantine/revocation carry measured prices** on strong reader (utility cost)
- **One task gate breaches on strong reader** — boundary is conditional, not absolute
- **Does not assert truth** — determines standing to influence action

## What this does NOT establish

- Perfect trust boundary (one breach on strong reader)
- Zero-cost trust checking (priced checks)
- Real-corpus extraction quality for trust evidence
- Multi-agent disagreement and shared memory trust

## Try it yourself

Create a memory with ambiguous provenance and see which stage rejects it. Modify the WorkFrame and observe how authority check changes.

In [ ]:
# TRY IT YOURSELF: the same pool under escalating policy levels shows
# where each adversarial unit falls, with its reason code.
fx = TF.eval_fixtures()[0]
units = to_units(fx)
task = TP.TaskPacket(task_id=fx.behavior_task, query=fx.query,
                     as_of=fx.as_of, project=fx.project,
                     caller_scope=fx.caller_scope,
                     consequential=fx.consequential)
print(f"Fixture {fx.fixture_id}, attack units: {fx.attack_units}\n")
for uid in fx.attack_units:
    path = []
    for level in ("S1", "S2", "S3", "FULL"):
        adm = TP.apply_policy(units, task, level).admissions[uid]
        path.append(f"{level}={adm.verdict}({adm.stage})")
    print(f"  {uid}: {' -> '.join(path)}")

## Where this leads next

Chapter 18 composes the earned architecture end-to-end. The trust gate is one of the conditional controls in the final system:

> **Trust and admission gate** — Standing to influence action; a core architectural boundary, implemented as a conditional control with priced checks.

> **See this chapter in code:** [Open the companion Jupyter notebook](memory\17-chapter.ipynb)